# 03 · Pipeline 2 - Random Forest con validación cruzada
**EduPredict** · Samsung Innovation Campus 2025 · Reto 4 · Universidad del Rosario

> **Responsable:** Johan A. Vera Lozano  
> **Objetivo:** Entrenar el clasificador numérico con CV estratificada 5-fold y análisis de importancia de features.

---

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src.preprocessing import preprocess
from src.models import build_rf, train_rf_with_cv, evaluate_model
from src.config import NUMERIC_FEATURES, CLASS_LABELS, CV_FOLDS

UR_RED = "#DA0921"; UR_NAVY = "#242839"; UR_TECH = "#0E6A8C"; UR_GREEN = "#1A6E3A"

data = preprocess("../data/evaluaciones_docentes.csv")
print(f"X_num_train: {data.X_num_train.shape}")
print("✅ Datos cargados")

## 1 · Hiperparámetros del Random Forest

In [ ]:
rf = build_rf()
print("Configuración del Random Forest:")
params = rf.get_params()
for k, v in params.items():
    print(f"  {k:<25}: {v}")

print("Configuracion del Random Forest:")
print("  n_estimators=300: estabilidad sin costo excesivo")
print("  max_depth=12: evita overfitting")
print("  min_samples_leaf=4: cada hoja necesita >=4 muestras para generalizar")
print("  class_weight=balanced: compensa desbalance 50/30/20")
print("  n_jobs=-1: usa todos los cores disponibles")

## 2 · Validación cruzada estratificada 5-fold

In [ ]:
rf_model, cv_results = train_rf_with_cv(data.X_num_train, data.y_train)

print("Resultados de validación cruzada (5-fold estratificada):")
print(f"  Accuracy  - media: {cv_results['cv_accuracy_mean']:.4f}  std: ±{cv_results['cv_accuracy_std']:.4f}")
print(f"  F1-macro  - media: {cv_results['cv_f1_macro_mean']:.4f}  std: ±{cv_results['cv_f1_macro_std']:.4f}")
print(f"\n  Accuracy por fold:  {[round(x, 4) for x in cv_results['cv_accuracy_per_fold']]}")
print(f"  F1-macro por fold:  {[round(x, 4) for x in cv_results['cv_f1_per_fold']]}")
print("\n📌 Baja varianza entre folds -> el modelo generaliza bien (no overfitting)")

In [ ]:
# Visualizar resultados por fold
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
folds = [f"Fold {i+1}" for i in range(CV_FOLDS)]

for ax, metric, vals, title in zip(
    axes,
    ["Accuracy", "F1-macro"],
    [cv_results["cv_accuracy_per_fold"], cv_results["cv_f1_per_fold"]],
    ["CV Accuracy por fold", "CV F1-macro por fold"],
):
    colors = [UR_RED if v == min(vals) else UR_TECH for v in vals]
    bars = ax.bar(folds, vals, color=colors, edgecolor="white", alpha=0.9)
    ax.axhline(np.mean(vals), color=UR_NAVY, linewidth=1.5,
               linestyle="--", label=f"Media: {np.mean(vals):.4f}")
    ax.set_ylim(0.85, 0.95); ax.legend(frameon=False)
    ax.set_title(title, fontsize=12, fontweight="bold", color=UR_NAVY)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f"{v:.4f}", ha="center", fontsize=9, color=UR_NAVY)

plt.tight_layout()
plt.savefig("../outputs/rf_cv_results.png", dpi=150, bbox_inches="tight")
plt.show()

## 3 · Evaluación en test

In [ ]:
rf_proba_test = rf_model.predict_proba(data.X_num_test)
rf_pred_test  = rf_model.predict(data.X_num_test)

metrics_rf = evaluate_model(
    data.y_test, rf_pred_test, rf_proba_test, model_name="Random Forest"
)
print(f"Accuracy:      {metrics_rf['accuracy']:.4f}")
print(f"F1-macro:      {metrics_rf['f1_macro']:.4f}")
print(f"AUC-ROC macro: {metrics_rf['auc_roc_macro']:.4f}")
print("\nF1 por clase:")
for cls, f1 in metrics_rf["f1_per_class"].items():
    print(f"  {cls:<12}: {f1:.4f}")

## 4 · Importancia de features

In [ ]:
importances = rf_model.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]
feat_names  = [NUMERIC_FEATURES[i] for i in sorted_idx]
feat_imp    = importances[sorted_idx]

fig, ax = plt.subplots(figsize=(10, 5))
colors = [UR_RED if i == 0 else UR_TECH if i == 1 else UR_NAVY for i in range(len(feat_names))]
bars = ax.barh(feat_names[::-1], feat_imp[::-1], color=colors[::-1], edgecolor="white", alpha=0.9)
ax.set_title("Importancia de features - Random Forest",
             fontsize=13, fontweight="bold", color=UR_NAVY)
ax.set_xlabel("Importancia (Gini)", color="#555555")
for bar, v in zip(bars, feat_imp[::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f"{v:.4f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("../outputs/rf_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nImportancia por feature:")
for name, imp in zip(feat_names, feat_imp):
    bar = "█" * int(imp * 200)
    print(f"  {name:<28} {imp:.4f}  {bar}")

In [ ]:
import pickle
with open("../models/rf_model.pkl", "wb") as f:
    pickle.dump(rf_model, f)
with open("../models/scaler.pkl", "wb") as f:
    pickle.dump(data.scaler, f)
print("✅ Random Forest y Scaler guardados en models/")